In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import GridSearchCV
import pandas as pd
import joblib
import numpy as np
import sklearn
from lib.tnn_wrapper import TNetWrapper

import warnings
warnings.filterwarnings("ignore")


sklearn.set_config(transform_output="pandas")
pd.set_option('display.max_colwidth', None)


transformer = joblib.load("transformer.pkl")
train_df = pd.read_csv("data/balanced_train_data.csv")
y = train_df['TARGET'].copy().astype(np.float32)
train_df = train_df.drop(columns=['TARGET'])
X_transformed = transformer.transform(train_df).astype(np.float32)

del train_df


param_grid = {
    "hidden_units": [16, 32],
    "learning_rate": [0.01, 0.001],
    "epochs": [10, 12]
}

grid = GridSearchCV(
    TNetWrapper(),
    param_grid=param_grid,
    scoring='roc_auc_ovr',
    n_jobs=-1
)

grid.fit(X_transformed, y)

del X_transformed, y

test_df = pd.read_csv("data/split_data_test.csv")
y_test = test_df['TARGET'].astype(np.float32)
X_test = test_df.drop(columns=['TARGET'])
X_transformed_test = transformer.transform(X_test).astype(np.float32)

del X_test

best_model = grid.best_estimator_

y_prob = best_model.predict_proba(X_transformed_test)
y_pred = (y_prob > 0.5).astype(int)

best_params = {
    "hidden_units":best_model.hidden_units,
    "epochs" : best_model.epochs,
    "learning_rate" : best_model.learning_rate,
    "batch_size" : best_model.batch_size
}

result = pd.DataFrame({
    'library': ["scikit-learn, tensorflow"],
    'algorithm': ['Multilayer-perceptron'],
    'hyerparamters': [str(best_params)],
    'Accuracy': [accuracy_score(y_test, y_pred)],
    'AUC': [roc_auc_score(y_test, y_prob)]
}, index=None)

print(result.T)

I0000 00:00:1776096871.085815  497169 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776096879.988710  497169 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776096887.377035  497359 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776096887.399026  497362 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776096887.401377  497361 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776096887.402458  497360 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776096889.656446  497362 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776096889.668307  497360 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00

KeyboardInterrupt: 